In [ ]:
# Setup: install everything this notebook needs, regardless of environment.
# (pandas/numpy/matplotlib often come preinstalled in Colab, but this makes the
# notebook self-contained even in a bare Python environment.)
!pip install -q pandas numpy matplotlib altair geopandas folium contextily shapely


In [ ]:
# Self-healing data fetch: if HouseholdIncome.csv / FHSZSRA_23_3.gdb aren't already
# sitting next to this notebook (e.g. you opened just the .ipynb from GitHub/Colab
# rather than downloading the whole repo), pull them from the project repo instead.
import os

if not os.path.exists('HouseholdIncome.csv') or not os.path.exists('FHSZSRA_23_3.gdb'):
    print('Local data files not found -- cloning them from the project repo...')
    !rm -rf /tmp/_lafires_data
    !git clone --depth 1 https://github.com/kalafejzo1/lafiresfinalproject.git /tmp/_lafires_data
    !cp /tmp/_lafires_data/HouseholdIncome.csv .
    !cp -r /tmp/_lafires_data/FHSZSRA_23_3.gdb .
    print('Done.')
else:
    print('Local data files found -- using those.')


In [ ]:
import pandas as pd
import geopandas as gpd
import altair as alt

Loading data (running locally in VS Code, not Colab)

This notebook originally used Colab's `files.upload()` picker. Running locally instead, so we point directly at the files already on disk / their public source URLs.

In [ ]:
#Pull Zillow ZIP-level data directly from their public research URL (same source used later for Butte County)
zillow_url = "https://files.zillowstatic.com/research/public_csvs/zhvi/Zip_zhvi_uc_sfrcondo_tier_0.33_0.67_sm_sa_month.csv"
zillow = pd.read_csv(zillow_url)

income = pd.read_csv("HouseholdIncome.csv")

In [ ]:
hazard = gpd.read_file(
    "FHSZSRA_23_3.gdb"
)

zip_shapes = gpd.read_file(
    "https://www2.census.gov/geo/tiger/TIGER2024/ZCTA520/tl_2024_us_zcta520.zip"
)

zillow['zip'] = zillow['RegionName'].astype(str).str.zfill(5)

print(zip_shapes.head())
print(zip_shapes.columns)

hazard.head()

The Fire Hazard Severity Zone dataset was provided as a polygon shapefile, while the Zillow and Census datasets were organized by ZIP code. To make these datasets comparable, we created a ZIP-level wildfire exposure metric.

First, Zillow data was filtered to Los Angeles County ZIP codes, which defined the geographic scope of our analysis. ZIP code boundary polygons were then obtained from the U.S. Census and filtered to match the ZIP codes present in the Zillow dataset.

Next, both the ZIP code boundaries and Fire Hazard Severity Zone polygons were projected into the California Albers coordinate system (EPSG:3310) to ensure accurate area calculations. We then performed a spatial intersection between ZIP code polygons and fire hazard polygons. This allowed us to determine how much land area within each ZIP code fell into each fire hazard category (Moderate, High, and Very High).

For each ZIP code, we calculated the total area classified as either High or Very High fire hazard and divided that value by the total land area of the ZIP code. This produced a wildfire exposure metric:

Wildfire Exposure=
Total ZIP Code Area /Area in High or Very High Hazard Zones
	​


The resulting variable, pct_high_very_high_hazard, ranges from 0 to 1 and represents the proportion of each ZIP code's land area that is exposed to elevated wildfire risk.

This preprocessing step was necessary because the original fire hazard data was spatial (polygon-based), whereas our housing and demographic datasets were ZIP-level tabular datasets. Creating a ZIP-level wildfire exposure measure allowed us to merge wildfire risk, housing values, and income data into a single dataset and directly test our hypotheses regarding the relationship between income, housing markets, and wildfire risk.

In [ ]:
hazard_col = "FHSZ_Description"

# 1. Clean Zillow ZIPs and filter to Los Angeles County
zillow['zip'] = zillow['RegionName'].astype(str).str.zfill(5)

zillow_la = zillow[
    zillow['CountyName'].str.contains("Los Angeles", case=False, na=False)
].copy()

la_zips = set(zillow_la['zip'])

# 2. Clean ZIP boundary ZIPs
zip_shapes['zip'] = zip_shapes['ZCTA5CE20'].astype(str).str.zfill(5)

zip_shapes_la = zip_shapes[
    zip_shapes['zip'].isin(la_zips)
].copy()

# 3. Project both layers into California Albers for accurate area calculations
hazard_3310 = hazard.to_crs("EPSG:3310")
zip_shapes_la_3310 = zip_shapes_la.to_crs("EPSG:3310")

# 4. Calculate total area of each ZIP code
zip_shapes_la_3310['zip_total_area'] = zip_shapes_la_3310.geometry.area

zip_area = zip_shapes_la_3310[['zip', 'zip_total_area']].copy()

# 5. Intersect ZIP polygons with fire hazard polygons
hazard_zip = gpd.overlay(
    zip_shapes_la_3310[['zip', 'geometry']],
    hazard_3310[[hazard_col, 'geometry']],
    how='intersection'
)

# 6. Calculate area of each hazard category within each ZIP
hazard_zip['hazard_area'] = hazard_zip.geometry.area

hazard_summary = (
    hazard_zip
    .groupby(['zip', hazard_col])['hazard_area']
    .sum()
    .reset_index()
)

# 7. Pivot hazard categories into columns
hazard_pivot = hazard_summary.pivot(
    index='zip',
    columns=hazard_col,
    values='hazard_area'
).fillna(0).reset_index()

# 8. Add High + Very High area
hazard_pivot['high_very_high_area'] = 0

for col in hazard_pivot.columns:
    if col in ['High', 'Very High']:
        hazard_pivot['high_very_high_area'] += hazard_pivot[col]

# 9. Merge in total ZIP area
hazard_pivot = hazard_pivot.merge(
    zip_area,
    on='zip',
    how='left'
)

# 10. Correct metric: % of ENTIRE ZIP area in High / Very High hazard zones
hazard_pivot['pct_high_very_high_hazard'] = (
    hazard_pivot['high_very_high_area'] / hazard_pivot['zip_total_area']
)

# 11. Keep final clean hazard table
hazard_clean = hazard_pivot[['zip', 'pct_high_very_high_hazard']].copy()

# 12. Sanity checks
print(hazard_clean.shape)
print(hazard_clean['pct_high_very_high_hazard'].describe())

hazard_clean.sort_values(
    'pct_high_very_high_hazard',
    ascending=False
).head(20)

Clean Zillow and create home value variables

In [ ]:
date_cols = [c for c in zillow_la.columns if c[:4].isdigit()]

latest_col = date_cols[-1]      # latest Zillow date
old_col = "2015-05-31"          # 10 years before 2025-05

zillow_clean = zillow_la[['zip', 'City', 'CountyName', latest_col, old_col]].copy()

zillow_clean = zillow_clean.rename(columns={
    latest_col: 'current_home_value',
    old_col: 'home_value_2015'
})

zillow_clean['appreciation_10yr'] = (
    zillow_clean['current_home_value'] - zillow_clean['home_value_2015']
) / zillow_clean['home_value_2015']

Clean Census income

In [ ]:
income_clean = income.copy()

income_clean['zip'] = income_clean['NAME'].str.extract(r'(\d{5})')

income_clean = income_clean.rename(columns={
    'B19013_001E': 'median_household_income'
})

income_clean['median_household_income'] = pd.to_numeric(
    income_clean['median_household_income'],
    errors='coerce'
)

income_clean = income_clean[['zip', 'median_household_income']]

Merge everything

In [ ]:
final_df = (
    zillow_clean
    .merge(income_clean, on='zip', how='left')
    .merge(hazard_clean, on='zip', how='left')
)

# ZIPs with no spatial overlap with any fire hazard polygon never get a row from the
# intersection step, so they come back as NaN here. Treat "no overlap" as 0% exposure.
final_df['pct_high_very_high_hazard'] = final_df['pct_high_very_high_hazard'].fillna(0)

final_df.head()

Inspect data before visualizing

In [ ]:
final_df[['zip',
          'median_household_income',
          'current_home_value',
          'appreciation_10yr',
          'pct_high_very_high_hazard']].head()

In [ ]:
final_df.isna().sum()

Hypothesis 1:
Higher-income ZIP codes experience lower wildfire exposure than lower-income ZIP codes.

Test hypothesis numerically

In [ ]:
eda_df = final_df.dropna(
    subset=['median_household_income', 'pct_high_very_high_hazard']
).copy()

eda_df[['median_household_income',
        'pct_high_very_high_hazard']].corr()

Visualize with scatter and linear regression

In [ ]:
scatter1 = alt.Chart(eda_df).mark_circle(
    size=60,
    opacity=0.5
).encode(
    x=alt.X(
        'median_household_income:Q',
        title='Median Household Income ($)'
    ),
    y=alt.Y(
        'pct_high_very_high_hazard:Q',
        title='% of ZIP in High/Very High Fire Hazard Zone'
    ),
    tooltip=[
        'zip',
        'City',
        'median_household_income',
        'pct_high_very_high_hazard'
    ]
).properties(
    title='Income vs Wildfire Exposure',
    width=700,
    height=450
)

scatter1

The initial numerical analysis and scatterplot do not support our hypothesis. Instead of observing lower wildfire exposure among higher-income ZIP codes, we found a weak positive relationship between income and wildfire exposure (r = 0.212). The scatterplot suggests that several of the highest-income ZIP codes also experience some of the highest levels of wildfire risk. This unexpected result motivated further exploration.

Creating income quartiles

In [ ]:
eda_df['income_group'] = pd.qcut(
    eda_df['median_household_income'],
    q=4,
    labels=[
        'Lowest Income',
        'Lower-Middle Income',
        'Upper-Middle Income',
        'Highest Income'
    ]
)

eda_df[['median_household_income', 'income_group']].head()

Creating boxplot

In [ ]:
income_order = [
    'Lowest Income',
    'Lower-Middle Income',
    'Upper-Middle Income',
    'Highest Income'
]

boxplot = alt.Chart(eda_df).mark_boxplot(size=50).encode(
    x=alt.X(
        'income_group:N',
        sort=income_order,
        title='Income Group'
    ),
    y=alt.Y(
        'pct_high_very_high_hazard:Q',
        title='% of ZIP in High / Very High Fire Hazard Zones'
    ),
    tooltip=[
        'income_group',
        alt.Tooltip('pct_high_very_high_hazard:Q', format='.2f')
    ]
).properties(
    title='Wildfire Exposure Across Income Groups',
    width=700,
    height=450
)

boxplot

Your highest-income group has the widest spread and reaches the highest wildfire exposure values. That does not support the hypothesis that higher-income ZIPs are less exposed to wildfire risk. It suggests some wealthy ZIPs are highly exposed, likely hillside/coastal areas like Malibu, Topanga, and Calabasas.

Colored Scatterplot

In [ ]:
alt.Chart(eda_df).mark_circle(
    size=150,
    opacity=0.75
).encode(
    x=alt.X(
        'median_household_income:Q',
        title='Median Household Income ($)'
    ),
    y=alt.Y(
        'pct_high_very_high_hazard:Q',
        title='% of ZIP in High / Very High Fire Hazard Zones'
    ),
    color=alt.Color(
        'current_home_value:Q',
        title='Current Home Value ($)',
        scale=alt.Scale(scheme='viridis')
    ),
    tooltip=[
        'zip',
        'City',
        alt.Tooltip('median_household_income:Q', format=',.0f'),
        alt.Tooltip('current_home_value:Q', format='$,.0f'),
        alt.Tooltip('pct_high_very_high_hazard:Q', format='.2f')
    ]
).properties(
    title='Income, Home Values, and Wildfire Exposure',
    width=700,
    height=500
)

This visualization incorporates a third variable, current home value, into the relationship between income and wildfire exposure. Color allows us to identify whether high-risk ZIP codes are also high-value real estate markets. The chart reveals that several ZIP codes with both high incomes and substantial wildfire exposure also have some of the highest housing values in Los Angeles County.

In [ ]:
import pandas as pd
import altair as alt

#Pull Zillow County-level Data directly from their public URL
#This bypasses the need for any local CSV uploads
zillow_url = "https://files.zillowstatic.com/research/public_csvs/zhvi/County_zhvi_uc_sfrcondo_tier_0.33_0.67_sm_sa_month.csv"
df = pd.read_csv(zillow_url)

#Filter for California
ca_df = df[df['State'] == 'CA'].copy()

#Focus on Butte County (Camp Fire, Nov 2018) vs. State Average
#We will look at Jan 2017 through early 2024 to see the before and after
date_cols = [c for c in ca_df.columns if c.startswith('2017') or c.startswith('2018') or
             c.startswith('2019') or c.startswith('2020') or c.startswith('2021') or
             c.startswith('2022') or c.startswith('2023') or c.startswith('2024')]

#Calculate CA average across all counties
ca_avg = ca_df[date_cols].mean()

#Get Butte County specifically
butte = ca_df[ca_df['RegionName'] == 'Butte County'][date_cols].mean()

#Combine for Altair plotting
plot_df = pd.DataFrame({
    'Date': pd.to_datetime(date_cols),
    'Butte County': butte.values,
    'California Average': ca_avg.values
}).melt(id_vars=['Date'], var_name='Region', value_name='Typical Home Value')

#Visualize the trend
line_chart = alt.Chart(plot_df).mark_line(size=3).encode(
    x=alt.X('Date:T', title='Year'),
    y=alt.Y('Typical Home Value:Q', scale=alt.Scale(zero=False), title='Zillow Home Value Index ($)'),
    color=alt.Color('Region:N', title='Area', scale=alt.Scale(scheme='set1')),
    tooltip=[alt.Tooltip('Date:T', format='%b %Y'), 'Region:N', alt.Tooltip('Typical Home Value:Q', format='$,.0f')]
).properties(
    title='Housing Value Recovery Post-2018 Camp Fire (Butte County vs. CA Average)',
    width=700,
    height=450
)

#Add a vertical line for the Camp Fire (November 8, 2018)
fire_date = pd.DataFrame({'Date': [pd.to_datetime('2018-11-08')]})
rule = alt.Chart(fire_date).mark_rule(color='red', strokeDash=[5, 5], size=2).encode(x='Date:T')

#Combine and show
(line_chart + rule)

alt.LayerChart(...)

In [ ]:
#Identify the Zillow date column corresponding to the fire (Nov 2018)
fire_month_col = [c for c in date_cols if '2018-11' in c][0]

#Baseline both series to 100 at the time of the fire
butte_indexed = (butte / butte[fire_month_col]) * 100
ca_avg_indexed = (ca_avg / ca_avg[fire_month_col]) * 100

#Prepare data for plotting
plot_idx_df = pd.DataFrame({
    'Date': pd.to_datetime(date_cols),
    'Butte County': butte_indexed.values,
    'California Average': ca_avg_indexed.values
}).melt(id_vars=['Date'], var_name='Region', value_name='Index Value')

#Filter to only show the timeline from the fire onward (Nov 2018 to early 2024)
plot_idx_df = plot_idx_df[plot_idx_df['Date'] >= '2018-10-01']

#Visualize the Indexed Trend
indexed_chart = alt.Chart(plot_idx_df).mark_line(size=3).encode(
    x=alt.X('Date:T', title='Year'),
    y=alt.Y('Index Value:Q', scale=alt.Scale(domain=[90, 160]), title='Indexed Home Value (Nov 2018 = 100)'),
    color=alt.Color('Region:N', title='Area', scale=alt.Scale(scheme='set1')),
    tooltip=[alt.Tooltip('Date:T', format='%b %Y'), 'Region:N', alt.Tooltip('Index Value:Q', format='.1f')]
).properties(
    title='Indexed Property Value Recovery Post-Camp Fire (Base 100)',
    width=700,
    height=450
)

#Combine with the fire date rule
(indexed_chart + rule)

alt.LayerChart(...)

In [ ]:
#Identify 1-year, 3-year, and 5-year post-fire date columns
yr1_col = [c for c in date_cols if '2019-11' in c][0]
yr3_col = [c for c in date_cols if '2021-11' in c][0]
yr5_col = [c for c in date_cols if '2023-11' in c][0]

#Calculate percentage changes for Butte and CA Average
def calc_appreciation(series, start_col, end_col):
    return ((series[end_col] - series[start_col]) / series[start_col]) * 100

butte_appr = {
    '1-Year': calc_appreciation(butte, fire_month_col, yr1_col),
    '3-Year': calc_appreciation(butte, fire_month_col, yr3_col),
    '5-Year': calc_appreciation(butte, fire_month_col, yr5_col)
}

ca_appr = {
    '1-Year': calc_appreciation(ca_avg, fire_month_col, yr1_col),
    '3-Year': calc_appreciation(ca_avg, fire_month_col, yr3_col),
    '5-Year': calc_appreciation(ca_avg, fire_month_col, yr5_col)
}

#Calculate Relative Performance (Butte - CA Average)
rel_perf = {
    '1-Year': butte_appr['1-Year'] - ca_appr['1-Year'],
    '3-Year': butte_appr['3-Year'] - ca_appr['3-Year'],
    '5-Year': butte_appr['5-Year'] - ca_appr['5-Year']
}

perf_df = pd.DataFrame(list(rel_perf.items()), columns=['Timeframe', 'Relative Appreciation (%)'])
perf_df['Performance'] = perf_df['Relative Appreciation (%)'].apply(lambda x: 'Outperform' if x > 0 else 'Underperform')

#Visualize Relative Performance
bar_chart = alt.Chart(perf_df).mark_bar().encode(
    y=alt.Y('Timeframe:N', sort=['1-Year', '3-Year', '5-Year'], title='Time Since Fire'),
    x=alt.X('Relative Appreciation (%):Q', title='Appreciation Relative to CA Average (%)'),
    color=alt.Color('Performance:N', scale=alt.Scale(domain=['Outperform', 'Underperform'], range=['#4c78a8', '#f58518'])),
    tooltip=['Timeframe', alt.Tooltip('Relative Appreciation (%):Q', format='.2f')]
).properties(
    title='Butte County Home Value Performance Relative to CA Average',
    width=700,
    height=300
)

bar_chart

alt.Chart(...)

Interactive Map: Putting the ZIP Codes on the Map

Every visualization so far (scatterplots, boxplot, colored scatter) treats ZIP codes as abstract points on an income/wildfire axis, with no sense of where they actually are in Los Angeles County. Since we already have ZIP boundary polygons (`zip_shapes_la`) from the spatial join used to build our wildfire exposure metric, we can reuse those same geometries to build an interactive choropleth map. This lets a user toggle between wildfire exposure, income, home value, and appreciation as the fill color, and hover over any ZIP to see all four metrics at once — giving geographic context that a scatterplot can't.

In [ ]:
import folium
from folium.features import GeoJsonTooltip

# --- Build a single GeoDataFrame combining ZIP boundaries with all ZIP-level metrics ---
map_df = zip_shapes_la[['zip', 'geometry']].merge(final_df, on='zip', how='inner')
map_df = map_df.set_geometry('geometry')

if map_df.crs is None or map_df.crs.to_epsg() != 4326:
    map_df = map_df.to_crs(epsg=4326)

map_df['wildfire_pct'] = map_df['pct_high_very_high_hazard'] * 100

map_df = map_df.dropna(subset=[
    'median_household_income', 'current_home_value',
    'appreciation_10yr', 'wildfire_pct'
])

# --- Base map centered on LA County ---
m = folium.Map(location=[34.05, -118.25], zoom_start=9, tiles='cartodbpositron')

metrics = [
    ('wildfire_pct', 'Wildfire Exposure (%)', 'YlOrRd'),
    ('median_household_income', 'Median Household Income ($)', 'BuGn'),
    ('current_home_value', 'Current Home Value ($)', 'PuBu'),
    ('appreciation_10yr', '10-Year Appreciation (fraction)', 'YlGnBu'),
]

choropleth_names = []
for i, (col, label, palette) in enumerate(metrics):
    choropleth = folium.Choropleth(
        geo_data=map_df,
        data=map_df,
        columns=['zip', col],
        key_on='feature.properties.zip',
        fill_color=palette,
        fill_opacity=0.75,
        line_opacity=0.2,
        legend_name=label,
        name=label,
        show=(i == 0),   # only the first metric is visible on load
        overlay=True,    # a real overlay, not a "base layer" -- keeps it from fighting the tile layer
    ).add_to(m)
    choropleth_names.append(choropleth.get_name())

    # Only keep one legend rendered at a time so they don't stack on screen
    if i != 0:
        for key in list(choropleth._children):
            if key.startswith('color_map'):
                del choropleth._children[key]

# --- Hover tooltip overlay: shows all metrics regardless of which layer is toggled on ---
tooltip_layer = folium.GeoJson(
    map_df,
    style_function=lambda x: {'fillOpacity': 0, 'color': 'transparent'},
    tooltip=GeoJsonTooltip(
        fields=['zip', 'City', 'median_household_income',
                'current_home_value', 'appreciation_10yr', 'wildfire_pct'],
        aliases=['ZIP:', 'City:', 'Median Income ($):', 'Home Value ($):',
                 '10-Yr Appreciation:', 'Wildfire Exposure (%):'],
        localize=True,
    ),
    name='ZIP details (hover)',
    overlay=True,
    control=True,
).add_to(m)

folium.LayerControl(collapsed=False).add_to(m)

# Enforce "only one metric visible at a time" via JS instead of Folium's base-layer mechanism
# (base layers are mutually exclusive with EACH OTHER *and* the tile layer, which was silently
# hiding the street basemap every time a different metric was picked).
exclusive_js = f"""
<script>
window.addEventListener('load', function() {{
(function() {{
    var exclusiveLayers = [{', '.join(choropleth_names)}];
    {m.get_name()}.on('overlayadd', function(e) {{
        if (exclusiveLayers.indexOf(e.layer) !== -1) {{
            exclusiveLayers.forEach(function(layer) {{
                if (layer !== e.layer && {m.get_name()}.hasLayer(layer)) {{
                    {m.get_name()}.removeLayer(layer);
                }}
            }});
        }}
    }});
}})();
}});
</script>
"""
m.get_root().html.add_child(folium.Element(exclusive_js))

m

What's informative about this view: The map makes the earlier finding concrete — the highest-income, highest-wildfire-exposure ZIP codes (Malibu, Topanga, Calabasas) are visibly clustered along the coastal and foothill edges of the county, not scattered randomly. Toggling between layers shows that wildfire exposure and home value follow a similar geographic footprint, while median income spreads out differently, which is exactly the mismatch our correlation analysis picked up on. This is also the closest thing to an actual "prototype interaction" for the mid-term demo: it has a layer toggle and hover tooltips a user can click through live during the presentation.

What could be improved: The four layers use different color scales and bin breaks, so "toggle and compare" still relies on the viewer remembering the previous layer's colors rather than seeing both at once. A future iteration could add a side-by-side small-multiples view, or a bivariate color scheme (e.g., income x wildfire exposure in one legend) to show the mismatch directly instead of requiring the user to switch layers back and forth. We could also overlay the Woolsey and Palisades fire perimeters on this same map to connect it to Hypotheses 2 and 3.

Numbered Highlight Pins: Top ZIP Codes to Watch

To make the map easier to read at a glance during the presentation, we add numbered pins (styled after city-guide "top spots" maps) marking the 6 ZIP codes that best capture our central finding from Hypothesis 1: high home values sitting in areas of high wildfire exposure. These are the ZIP codes an investor or homebuyer would most want to interrogate further — high current value, but also high fire risk.

In [ ]:
import warnings
warnings.filterwarnings('ignore', category=UserWarning)

# --- Rank ZIPs by wildfire exposure, restricted to above-median home value ---
# This surfaces the "wealthy but exposed" ZIP codes our Hypothesis 1 analysis found
highlight_df = (
    map_df[map_df['current_home_value'] > map_df['current_home_value'].median()]
    .sort_values('wildfire_pct', ascending=False)
    .head(6)
    .reset_index(drop=True)
)
highlight_df['rank'] = highlight_df.index + 1

for _, row in highlight_df.iterrows():
    centroid = row.geometry.centroid

    pin_html = f"""
    <div style="
        background-color:#8e24aa;
        border-radius:50%;
        width:30px; height:30px;
        display:flex; align-items:center; justify-content:center;
        color:white; font-weight:bold; font-size:14px;
        border:2px solid white;
        box-shadow:0 2px 5px rgba(0,0,0,0.4);">
        {row['rank']}
    </div>
    """

    popup_html = f"""
    <b>#{row['rank']} — {row['City']} ({row['zip']})</b><br>
    Wildfire Exposure: {row['wildfire_pct']:.1f}%<br>
    Home Value: ${row['current_home_value']:,.0f}<br>
    Median Income: ${row['median_household_income']:,.0f}
    """

    folium.Marker(
        location=[centroid.y, centroid.x],
        icon=folium.DivIcon(html=pin_html),
        popup=folium.Popup(popup_html, max_width=250),
        tooltip=f"#{row['rank']}: {row['City']}",
    ).add_to(m)

m

In [ ]:
# Save as a standalone HTML file — VS Code's notebook renderer can be finicky about
# raw HTML/JS outputs like Folium maps, so opening this file directly in a browser
# is the most reliable way to view/demo it (also better for a live presentation).
m.save("la_wildfire_map.html")
print("Saved to la_wildfire_map.html — open it directly in your browser")

Bivariate Hex Map: A Different Visual Language

The interactive map above already flagged one limitation: comparing wildfire exposure and home value means switching between layers, since each uses its own color scale. Inspired by bivariate dot-density maps used in climate reporting (e.g. drought-intensity maps, where color encodes severity and dot size encodes frequency), this static hex-mosaic map encodes both variables at once: color = wildfire exposure, size = current home value. Each hexagon is a small sample point spatially joined to its containing ZIP polygon, so the mosaic naturally follows the real shape of LA County rather than showing one blob per ZIP.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import RegularPolygon
from matplotlib.collections import PatchCollection
from matplotlib.colors import LinearSegmentedColormap, Normalize
from shapely.geometry import Point
import contextily as cx

hex_df = map_df.to_crs('EPSG:3310')
minx, miny, maxx, maxy = hex_df.total_bounds

# --- Build a fine hex lattice covering the bounding box ---
dx = 1900
dy = dx * np.sqrt(3) / 2

xs = np.arange(minx - dx, maxx + dx, dx)
centers = []
row = 0
y = miny - dy
while y < maxy + dy:
    x_offset = (dx / 2) if (row % 2) else 0
    for x in xs + x_offset:
        centers.append((x, y))
    y += dy
    row += 1

pts = gpd.GeoDataFrame(geometry=[Point(c) for c in centers], crs='EPSG:3310')
pts['hx'] = [c[0] for c in centers]
pts['hy'] = [c[1] for c in centers]

# Assign each hex point the wildfire/home-value stats of the ZIP polygon it falls inside
joined = gpd.sjoin(
    pts, hex_df[['zip', 'wildfire_pct', 'current_home_value', 'geometry']],
    how='inner', predicate='within'
)
print(f"Hex points total: {len(pts)}, matched inside LA ZIPs: {len(joined)}")

jminx, jminy, jmaxx, jmaxy = joined['hx'].min(), joined['hy'].min(), joined['hx'].max(), joined['hy'].max()
pad = 4000
jminx, jminy, jmaxx, jmaxy = jminx - pad, jminy - pad, jmaxx + pad, jmaxy + pad

cmap = LinearSegmentedColormap.from_list(
    'wildfire',
    ['#f7ecd1', '#f7c948', '#f2822a', '#e8412c', '#c31e5c', '#5b1a8c'],
)
vmax = joined['wildfire_pct'].quantile(0.98)
color_norm = Normalize(vmin=0, vmax=vmax)

val = joined['current_home_value'].clip(upper=joined['current_home_value'].quantile(0.98))
vmin_, vmax_ = val.min(), val.max()
size_norm = (np.sqrt(val) - np.sqrt(vmin_)) / (np.sqrt(vmax_) - np.sqrt(vmin_))
radius = 550 + size_norm.values * 1450

fig, ax = plt.subplots(figsize=(11, 11.5), dpi=220)
fig.patch.set_facecolor('#f2ede2')

patches = []
colors = []
for (x, y), r, c in zip(joined[['hx', 'hy']].values, radius, joined['wildfire_pct'].values):
    patches.append(RegularPolygon((x, y), numVertices=6, radius=r, orientation=0))
    colors.append(c)

pc = PatchCollection(patches, cmap=cmap, norm=color_norm)
pc.set_array(np.array(colors))
pc.set_edgecolor('none')
pc.set_alpha(0.85)
ax.add_collection(pc)

ax.set_xlim(jminx, jmaxx)
ax.set_ylim(jminy, jmaxy)
ax.set_aspect('equal')
ax.axis('off')

# Real street basemap underneath the hex dots (reproject tiles into our CA Albers CRS)
cx.add_basemap(ax, crs=hex_df.crs.to_string(), source=cx.providers.CartoDB.Positron)

# --- City labels ---
city_labels = {
    'Malibu': (-118.7798, 34.0259),
    'Santa Monica': (-118.4912, 34.0195),
    'Downtown LA': (-118.2437, 34.0407),
    'Pasadena': (-118.1445, 34.1478),
    'Long Beach': (-118.1937, 33.7701),
    'Santa Clarita': (-118.5426, 34.3917),
}
labels_gdf = gpd.GeoDataFrame(
    {'name': list(city_labels.keys())},
    geometry=[Point(v) for v in city_labels.values()],
    crs='EPSG:4326',
).to_crs('EPSG:3310')

for name, geom in zip(labels_gdf['name'], labels_gdf.geometry):
    if not (jminx <= geom.x <= jmaxx and jminy <= geom.y <= jmaxy):
        continue
    ax.plot(geom.x, geom.y, marker='o', markersize=3.5, color='#2b2118', zorder=5)
    ax.annotate(name, (geom.x, geom.y), xytext=(6, 4), textcoords='offset points',
                fontsize=10, color='#2b2118', fontweight='medium', zorder=5)

# --- Annotated callouts (leader-line style, like the reference chart) ---
def callout(zip_code, text, xytext, ha='left'):
    row = hex_df[hex_df['zip'] == zip_code].iloc[0]
    pt = row.geometry.representative_point()
    ax.annotate(
        text, xy=(pt.x, pt.y), xytext=xytext, textcoords='offset points',
        fontsize=9.3, color='#2b2118', ha=ha, va='center',
        arrowprops=dict(arrowstyle='-', color='#2b2118', lw=0.9,
                        connectionstyle='arc3,rad=0.15'),
        bbox=dict(boxstyle='round,pad=0.35', fc='#f2ede2', ec='none', alpha=0.85),
        zorder=6,
    )
    ax.plot(pt.x, pt.y, marker='o', markersize=4, mfc='none', mec='#2b2118', mew=1.2, zorder=6)

callout('90290', 'Topanga: 94% of the ZIP sits in\nhigh/very-high fire hazard zones —\nthe most exposed ZIP in LA County',
        (140, 90))
callout('90265', 'Malibu: $3.19M median home value\ndespite 59% wildfire exposure',
        (150, -70))
callout('91307', '"West Hills" — labeled just "Los Angeles"\nin Zillow\'s data — has 47% wildfire\nexposure and a $1.0M median home value',
        (-260, -40), ha='right')

# --- Bivariate diamond legend (own inset axes, own coordinate system) ---
lax = fig.add_axes([0.065, 0.70, 0.20, 0.20])
lax.set_xlim(-1, 11)
lax.set_ylim(-2.2, 9)
lax.axis('off')
lax.set_facecolor('none')

n = 4
step_l = 1.55
for i in range(n):
    for j in range(n):
        lx = 5 + (i - j) * step_l * 0.62
        ly = (i + j) * step_l * 0.5
        c_val = (i / (n - 1)) * vmax
        s_val = 0.30 + (j / (n - 1)) * 0.42
        lax.add_patch(RegularPolygon((lx, ly), numVertices=6, radius=s_val,
                                      orientation=0, facecolor=cmap(color_norm(c_val)),
                                      edgecolor='none'))

lax.text(5, 9.0, 'Bivariate key', fontsize=9.5, color='#2b2118', ha='center', fontweight='medium')
lax.text(1.4, -1.3, 'More wildfire\nexposure', fontsize=8.3, color='#2b2118', ha='center', va='top')
lax.text(8.6, -1.3, 'Higher\nhome value', fontsize=8.3, color='#2b2118', ha='center', va='top')

# --- Title ---
fig.text(0.065, 0.955, 'Wildfire Exposure & Home Values Across LA County',
          fontsize=21, color='#2b2118', fontfamily='serif')
fig.text(0.065, 0.932,
         'Color = share of ZIP in high/very-high fire hazard zone   |   Size = current median home value',
         fontsize=11.5, color='#6b5d4f')

plt.subplots_adjust(top=0.90)
plt.savefig("la_wildfire_hexmap.png", bbox_inches='tight', facecolor=fig.get_facecolor())
plt.show()

What's informative about this view: Encoding both variables at once — color for wildfire exposure, size for home value — makes the Hypothesis 1 finding visible in a single glance rather than requiring a layer toggle: the Malibu/Topanga/West Hills corridor along the county's western edge shows up as large, dark, saturated hexagons — high value and high risk in the same place. It also surfaces the ZIP-labeling issue directly on the map: neighborhoods like West Hills and Chatsworth get folded into a generic "Los Angeles" label in Zillow's data despite being distinct, well-known areas with very different wildfire exposure than downtown.

What could be improved: The bivariate encoding is powerful but has a steeper learning curve than a single-variable choropleth — the diamond legend takes a few seconds to parse the first time. It's also static, so it loses the hover-for-exact-values interactivity of the Folium map above. A combined approach — this bivariate encoding rendered as an interactive layer with tooltips — would likely be the strongest version for a final deliverable.

Bringing It Together: The LA Wildfire Risk & Investment Explorer

Our proposal set out to let users "explore wildfire activity alongside housing market, insurance, and demographic data" and to "identify whether some communities continue to experience strong housing growth despite wildfire risk, while others show signs of increased vulnerability." This version reframes the earlier dynamic map explicitly around that real-estate decision — the layer names, legend, and pins below are written for an investor or homebuyer, not just for our own analysis. It combines:

- **5 continuous ZIP-level layers**, still toggleable: Wildfire Risk, Median Household Income, Median Home Value, 10-Year Price Appreciation, and a new **Investment Opportunity Score** — each ZIP's percentile rank on 10-year appreciation minus its percentile rank on wildfire exposure, weighted equally, so a ZIP can't top the ranking purely on growth the way a raw-value formula would let it. It is based on wildfire risk, historical appreciation, and housing market characteristics, and is intended as a screening tool for further due diligence, not a definitive investment recommendation.
- **An "Investment Category" layer** — the same risk/appreciation quadrant as before, relabeled in investor-facing language: **Opportunity Zone** (fire-exposed but still appreciating above the county median), **Red Flag** (fire-exposed without the appreciation to match), **Safe Bet**, and **Low Priority**.
- **Numbered "Top Investment Picks" pins** marking the 8 ZIPs with the highest Investment Score, each with a popup summarizing home value, appreciation, wildfire risk, and category — a concrete, ranked answer to "where should I look first?"
- **Historical major fires** (CAL FIRE FRAP's public statewide dataset, filtered to fires ≥3,000 acres near LA County) as a toggleable overlay of bubbles sized by acres burned and colored by decade, including Woolsey (2018, ~89,551 acres) and Palisades (2025, ~23,449 acres) — so a user can check whether a "Top Pick" ZIP has actually burned before, not just whether it's modeled as hazardous. (An earlier version drew these as filled fire perimeters, but overlapping polygons stacked into a muddy mess — bubbles read far more cleanly.)

**Not yet included:** insurance availability and cost data (e.g., the California FAIR Plan enrollment data named in our proposal). We haven't sourced and cleaned that dataset yet, so rather than guess at insurance figures, this map leaves that layer out — for a real real-estate tool, insurance cost/availability would likely change the Investment Score more than any other single input, since a ZIP can look like a "Safe Bet" on appreciation alone while quietly becoming uninsurable.

In [ ]:
import os
import urllib.request
import numpy as np
from shapely.geometry import Point
from matplotlib.colors import LinearSegmentedColormap, Normalize
import matplotlib as mpl

# --- Fetch CAL FIRE FRAP's public historical fire perimeter dataset (cached locally: ~250MB, one-time) ---
FIRE_CACHE = "ca_fire_perimeters.geojson"
FIRE_URL = "https://hub.arcgis.com/api/download/v1/items/c3c10388e3b24cec8a954ba10458039d/geojson?redirect=true&layers=0"

if not os.path.exists(FIRE_CACHE):
    print("Downloading CAL FIRE historical fire perimeters (one-time, ~250MB)...")
    urllib.request.urlretrieve(FIRE_URL, FIRE_CACHE)

la_bounds_3857 = map_df.to_crs(epsg=3857).total_bounds
buf = 20000
bbox = (la_bounds_3857[0] - buf, la_bounds_3857[1] - buf, la_bounds_3857[2] + buf, la_bounds_3857[3] + buf)

fires = gpd.read_file(FIRE_CACHE, bbox=bbox).to_crs(epsg=4326)
fires = fires[fires['GIS_ACRES'] >= 3000].copy()
fires = fires.dropna(subset=['DECADES'])
fires['YEAR_'] = fires['YEAR_'].astype('Int64')
fires['acres_fmt'] = fires['GIS_ACRES'].round(0).astype(int)
fires['fire_lat'] = fires.geometry.centroid.y
fires['fire_lon'] = fires.geometry.centroid.x
print(f"Historical fires kept (>=3,000 acres, near LA County): {len(fires)}")

# --- Investment Opportunity Score: risk-adjusted appreciation ---
# A simple, transparent illustrative score (NOT a real underwriting metric, and not
# investment advice -- intended as a screening tool for further due diligence).
#
# Earlier version: appreciation_10yr / (1 + wildfire_pct / 100). That divisor is
# bounded between 1 and ~1.94 across our whole dataset (max observed wildfire_pct is
# ~94%), so even the highest-risk ZIP in LA County only had its score cut roughly in
# half, while appreciation_10yr itself ranges from -0.09 to 1.85 -- a much wider span.
# Net effect: correlation between the old score and raw appreciation was 0.96 (and
# only -0.27 with wildfire risk), so growth was doing almost all the work and 7 of
# the "top 8" picks had exactly 0% measured wildfire exposure.
#
# Fix: percentile-rank both components onto the same 0-100 scale first, so neither
# can dominate just by having a wider raw numeric range, then combine with explicit,
# adjustable weights. (method='min' means every ZIP tied at 0% wildfire exposure gets
# the single best -- not average -- risk percentile, rather than being penalized
# relative to each other for a tie at the safest possible value.)
GROWTH_WEIGHT = 0.5
RISK_WEIGHT = 0.5

growth_percentile = map_df['appreciation_10yr'].rank(method='min', pct=True) * 100
risk_percentile = map_df['wildfire_pct'].rank(method='min', pct=True) * 100

map_df['investment_score'] = (
    GROWTH_WEIGHT * growth_percentile - RISK_WEIGHT * risk_percentile
)
# Ranges roughly -50 (worst: max risk, min growth) to +50 (best: min risk, max growth)

# --- Investment Category (same risk/growth quadrant as before, real-estate-facing labels) ---
# wildfire_pct is heavily zero-inflated (most ZIPs have 0% exposure), so a median split on
# risk is degenerate. Use "any measurable hazard-zone overlap" as the risk cutoff instead.
med_growth = map_df['appreciation_10yr'].median()

def classify(row):
    hi_risk = row['wildfire_pct'] > 0
    hi_growth = row['appreciation_10yr'] >= med_growth
    if hi_risk and hi_growth:
        return 'Opportunity Zone (high risk, high appreciation)'
    if hi_risk and not hi_growth:
        return 'Red Flag (high risk, low appreciation)'
    if not hi_risk and hi_growth:
        return 'Safe Bet (low risk, high appreciation)'
    return 'Low Priority (low risk, low appreciation)'

map_df['investment_category'] = map_df.apply(classify, axis=1)
print(map_df['investment_category'].value_counts())

category_colors = {
    'Opportunity Zone (high risk, high appreciation)': '#841c79',
    'Red Flag (high risk, low appreciation)': '#df3838',
    'Safe Bet (low risk, high appreciation)': '#f2892d',
    'Low Priority (low risk, low appreciation)': '#f7de9b',
}

# --- Build a hex-grid of sample points over LA County (in meters), then reproject to lon/lat ---
map_df_m = map_df.to_crs(epsg=3310)
minx, miny, maxx, maxy = map_df_m.total_bounds
dx = 2400
dy = dx * np.sqrt(3) / 2
xs = np.arange(minx - dx, maxx + dx, dx)
centers = []
row_i = 0
y = miny - dy
while y < maxy + dy:
    x_offset = (dx / 2) if (row_i % 2) else 0
    for x in xs + x_offset:
        centers.append((x, y))
    y += dy
    row_i += 1

pts = gpd.GeoDataFrame(geometry=[Point(c) for c in centers], crs='EPSG:3310')
joined_m = gpd.sjoin(pts, map_df_m[['zip', 'geometry']], how='inner', predicate='within')
hex_pts = gpd.GeoDataFrame(joined_m[['zip']], geometry=joined_m.geometry, crs='EPSG:3310').to_crs(epsg=4326)
hex_pts = hex_pts.merge(
    map_df[['zip', 'City', 'wildfire_pct', 'median_household_income', 'current_home_value',
            'appreciation_10yr', 'investment_score', 'investment_category']],
    on='zip', how='left'
)
hex_pts['lon'] = hex_pts.geometry.x
hex_pts['lat'] = hex_pts.geometry.y
print(f"Hex points: {len(hex_pts)}")

cmap = LinearSegmentedColormap.from_list(
    'wildfire', ['#f7ecd1', '#f7c948', '#f2822a', '#e8412c', '#c31e5c', '#5b1a8c']
)

def hex_color(value, norm):
    return mpl.colors.to_hex(cmap(norm(value)))

# --- Base map: cream background instead of map tiles, to match the bivariate hex map's look ---
dyn_map = folium.Map(location=[34.05, -118.25], zoom_start=9, prefer_canvas=True, tiles=None)
folium.TileLayer(
    tiles='cartodbpositron', name='Streets Basemap (CARTO)',
    attr='Basemap by CARTO, data by OpenStreetMap contributors', control=True,
).add_to(dyn_map)

metrics = [
    ('wildfire_pct', 'Wildfire Risk (%)'),
    ('median_household_income', 'Median Household Income ($)'),
    ('current_home_value', 'Median Home Value ($)'),
    ('appreciation_10yr', '10-Year Price Appreciation (Fraction)'),
    ('investment_score', 'Investment Opportunity Score'),
]

exclusive_fg_names = []
for i, (col, label) in enumerate(metrics):
    vmin = hex_pts[col].quantile(0.02)
    vmax = hex_pts[col].quantile(0.98)
    norm = Normalize(vmin=vmin, vmax=vmax)
    fg = folium.FeatureGroup(name=label, show=(i == 0), overlay=True)
    rmin, rmax = 2, 6.5
    for _, r in hex_pts.iterrows():
        v = r[col]
        radius = rmin + np.clip((v - vmin) / (vmax - vmin), 0, 1) * (rmax - rmin)
        folium.CircleMarker(
            location=(r['lat'], r['lon']),
            radius=radius,
            color='white',
            weight=0.8,
            fill=True,
            fill_color=hex_color(v, norm),
            fill_opacity=0.88,
        ).add_to(fg)
    fg.add_to(dyn_map)
    exclusive_fg_names.append(fg.get_name())

# --- Wildfire Exposure background texture (same muted treatment as the bubble map) ---
# An optional, always-available context layer, independent of whichever metric/category is selected
texture_layer = folium.FeatureGroup(name='Wildfire Exposure (background texture)', show=False, overlay=True)
tex_norm = Normalize(vmin=0, vmax=hex_pts['wildfire_pct'].quantile(0.95))
for _, r in hex_pts.iterrows():
    v = min(r['wildfire_pct'], hex_pts['wildfire_pct'].quantile(0.95))
    radius = 1.2 + tex_norm(v) * 3.5
    folium.CircleMarker(
        location=(r['lat'], r['lon']), radius=radius, color=None, fill=True,
        fill_color='#5b7ca8', fill_opacity=0.35, weight=0,
    ).add_to(texture_layer)
texture_layer.add_to(dyn_map)

# --- Investment Category layer (categorical dots, same radio group as the 5 metrics above) ---
fg_cat = folium.FeatureGroup(name='Investment Category', show=False, overlay=True)
for _, r in hex_pts.iterrows():
    folium.CircleMarker(
        location=(r['lat'], r['lon']),
        radius=4.5,
        color='white',
        weight=0.7,
        fill=True,
        fill_color=category_colors[r['investment_category']],
        fill_opacity=0.88,
    ).add_to(fg_cat)
fg_cat.add_to(dyn_map)
exclusive_fg_names.append(fg_cat.get_name())

# --- Hover tooltip overlay: property-card style with everything an investor would want ---
map_df['investment_score_fmt'] = map_df['investment_score'].round(2)
folium.GeoJson(
    map_df,
    style_function=lambda x: {'fillOpacity': 0, 'color': 'transparent'},
    tooltip=GeoJsonTooltip(
        fields=['zip', 'City', 'investment_category', 'current_home_value',
                'appreciation_10yr', 'wildfire_pct', 'investment_score_fmt'],
        aliases=['ZIP:', 'City:', 'Category:', 'Home Value ($):',
                 '10-Yr Appreciation:', 'Wildfire Risk (%):', 'Investment Score:'],
        localize=True,
    ),
    name='ZIP details (hover)',
    overlay=True,
    control=True,
).add_to(dyn_map)

# --- Historical fires overlay: bubbles (size = acres, color = decade), not filled perimeters ---
# Filled polygons looked messy — overlapping fire boundaries stack their opacity into muddy
# blobs and clash with the category dot colors underneath. Bubbles (same treatment as the
# editorial bubble map) show the same information without the visual noise.
fire_layer = folium.FeatureGroup(name='Historical Major Fires (>=3,000 acres, CAL FIRE FRAP)', show=False, overlay=True)
for _, r in fires.iterrows():
    radius = max(3, np.sqrt(r['GIS_ACRES']) * 0.045)
    popup_html = f"<b>{r['FIRE_NAME']}</b> ({int(r['YEAR_'])})<br>{r['acres_fmt']:,} acres burned"
    folium.CircleMarker(
        location=(r['fire_lat'], r['fire_lon']),
        radius=radius,
        color='#b3401f',
        weight=1,
        fill=True,
        fill_color='#f2d9a1',
        fill_opacity=0.85,
        popup=folium.Popup(popup_html, max_width=220),
        tooltip=f"{r['FIRE_NAME']} ({int(r['YEAR_'])}) — {r['acres_fmt']:,} acres",
    ).add_to(fire_layer)
fire_layer.add_to(dyn_map)

# --- City labels (text only, matching the static hex map's style) ---
city_labels = {
    'Malibu': (-118.7798, 34.0259), 'Santa Monica': (-118.4912, 34.0195),
    'Downtown LA': (-118.2437, 34.0407), 'Pasadena': (-118.1445, 34.1478),
    'Long Beach': (-118.1937, 33.7701), 'Santa Clarita': (-118.5426, 34.3917),
}
for name, (lon, lat) in city_labels.items():
    folium.Marker(
        location=(lat, lon),
        icon=folium.DivIcon(html=f'<div style="font-size:11px;color:#2b2118;font-weight:600;white-space:nowrap;">{name}</div>'),
    ).add_to(dyn_map)

# --- Top Investment Picks: numbered pins ranked by Investment Score ---
top_picks = map_df.sort_values('investment_score', ascending=False).head(8).reset_index(drop=True)
top_picks['rank'] = top_picks.index + 1
for _, row in top_picks.iterrows():
    centroid = row.geometry.centroid
    pin_html = f"""
    <div style="width:34px; height:34px;">
      <div style="
          width:26px; height:26px;
          background:#f5e6c8;
          border:2px solid #8a6d1f;
          border-radius:50% 50% 50% 0;
          transform:rotate(-45deg);
          box-shadow:0 3px 6px rgba(0,0,0,0.35);
          display:flex; align-items:center; justify-content:center;">
        <span style="transform:rotate(45deg); color:#3a2e1a; font-weight:bold; font-size:12.5px;">
          {row['rank']}
        </span>
      </div>
    </div>
    """
    popup_html = f"""
    <b>#{row['rank']} — {row['City']} ({row['zip']})</b><br>
    <span style="color:{category_colors[row['investment_category']]};">&#9632;</span> {row['investment_category']}<br>
    Home Value: ${row['current_home_value']:,.0f}<br>
    10-Yr Appreciation: {row['appreciation_10yr']*100:.1f}%<br>
    Wildfire Risk: {row['wildfire_pct']:.1f}%<br>
    Investment Score: {row['investment_score']:.2f}
    """
    folium.Marker(
        location=[centroid.y, centroid.x],
        icon=folium.DivIcon(html=pin_html, icon_size=(34, 34), icon_anchor=(17, 30)),
        popup=folium.Popup(popup_html, max_width=250),
        tooltip=f"#{row['rank']}: {row['City']} — Investment Score {row['investment_score']:.2f}",
    ).add_to(dyn_map)

# --- Masthead ---
masthead_html = '''
<div style="position: fixed; top: 0; left: 0; right: 0; z-index: 9999;
            background: white; padding: 14px 20px 10px 20px;
            box-shadow: 0 2px 6px rgba(0,0,0,0.15); font-family: sans-serif;">
  <div style="color:#b3401f; font-weight:bold; font-size:12px; letter-spacing:0.5px;">FOCUS &nbsp;|&nbsp; REAL ESTATE INVESTMENT</div>
  <div style="font-family: Georgia, serif; font-weight:bold; font-size:24px; color:#1a1a1a; margin-top:2px;">
    LA Wildfire Risk &amp; Investment Explorer
  </div>
  <div style="font-size:12.5px; color:#4a4a4a; margin-top:4px; max-width:780px; line-height:1.45;">
    Built for investors and homebuyers weighing whether a ZIP code's price growth justifies its wildfire exposure.
    Toggle layers to compare risk, income, home value, appreciation, and our Investment Opportunity Score
    (ZIP percentile rank on 10-yr appreciation, minus ZIP percentile rank on wildfire exposure, weighted equally).
    The Investment Opportunity Score is based on wildfire risk, historical appreciation, and housing market
    characteristics. It is intended as a screening tool for further due diligence, not a definitive investment
    recommendation. Numbered pins mark the 8 highest-scoring ZIPs; toggle on Historical Major Fires (cream circles,
    sized by acres burned) to see real burn history, or Wildfire Exposure for background context.
  </div>
</div>
'''
dyn_map.get_root().html.add_child(folium.Element(masthead_html))
dyn_map.get_root().html.add_child(folium.Element(
    "<style>.folium-map { margin-top: 148px; height: calc(100% - 148px) !important; }</style>"
))

# --- Fixed legend for the investment category layer ---
legend_html = '''
<div style="position: fixed; bottom: 30px; left: 30px; z-index: 9999;
            background: white; padding: 10px 14px; border-radius: 6px;
            box-shadow: 0 1px 6px rgba(0,0,0,0.3); font-size: 12.5px; line-height: 1.5;">
<b>Investment Category</b><br>
<span style="color:#841c79;">&#9632;</span> Opportunity Zone (high risk, high appreciation)<br>
<span style="color:#df3838;">&#9632;</span> Red Flag (high risk, low appreciation)<br>
<span style="color:#f2892d;">&#9632;</span> Safe Bet (low risk, high appreciation)<br>
<span style="color:#f7de9b;">&#9632;</span> Low Priority (low risk, low appreciation)<br>
<i style="color:#6b5d4f;">(only shown when that layer is active)</i>
<hr style="margin:6px 0;">
<span style="display:inline-block;width:10px;height:10px;border-radius:50%;
             background:#f2d9a1;border:1px solid #b3401f;margin-right:4px;"></span>
Historical major fire (size = acres burned)
</div>
'''
dyn_map.get_root().html.add_child(folium.Element(legend_html))

folium.LayerControl(collapsed=False).add_to(dyn_map)

# Enforce "only one metric/category visible at a time" via JS instead of Folium's base-layer
# mechanism (base layers are mutually exclusive with EACH OTHER *and* the tile layer, which was
# silently hiding the street basemap every time a different metric was picked).
exclusive_js = f"""
<script>
window.addEventListener('load', function() {{
(function() {{
    var exclusiveLayers = [{', '.join(exclusive_fg_names)}];
    {dyn_map.get_name()}.on('overlayadd', function(e) {{
        if (exclusiveLayers.indexOf(e.layer) !== -1) {{
            exclusiveLayers.forEach(function(layer) {{
                if (layer !== e.layer && {dyn_map.get_name()}.hasLayer(layer)) {{
                    {dyn_map.get_name()}.removeLayer(layer);
                }}
            }});
        }}
    }});
}})();
}});
</script>
"""
dyn_map.get_root().html.add_child(folium.Element(exclusive_js))

dyn_map.save("la_wildfire_dynamic_map.html")
print("Saved to la_wildfire_dynamic_map.html — open it directly in your browser")
dyn_map

What's informative about this view: Framing this around an actual real-estate decision changes what the map is for — instead of asking a user to compare five layers and draw their own conclusion, the Investment Score and numbered pins hand them a ranked, actionable answer. The 8 "Top Picks" are all ZIPs where 10-year appreciation outpaced the wildfire-exposure penalty, which is a very different (and more useful) framing than just "here's the wildfire map" and "here's the price map" side by side. Toggling on the historical fire bubbles over a Top Pick is a genuinely useful gut-check: several of the highest-scoring ZIPs sit right next to real Woolsey/Palisades fire locations, which the score alone doesn't show.

What could be improved: We originally used appreciation ÷ (1 + wildfire%), but that divisor only ranges from 1 to ~1.94 across our data while appreciation itself spans a much wider range, so growth alone explained 96% of the variance in the score and wildfire risk barely mattered (confirmed when a reviewer noticed a ZIP's score matched its raw appreciation number exactly). We replaced it with an equal-weighted percentile-rank formula instead, which gives both components comparable influence by construction — but the 50/50 weighting is still a judgment call, not validated against how real investors or lenders actually price wildfire risk, so it should be presented as illustrative, not authoritative, in the live demo. It also still can't speak to insurance at all: a ZIP could rank as a "Top Pick" on this score while quietly losing FAIR Plan coverage, which would matter enormously to a real buyer and isn't visible anywhere in this notebook yet. Sourcing and joining that data, and testing whether it changes the ranking, is the clearest next step before treating this as a real product rather than a class project prototype.

Editorial Bubble Map: Pinpointing Fire Exposure

Every map so far treats wildfire risk as a ZIP-level attribute. This view shifts the unit of analysis to the fire *events* themselves, styled after data-journalism graphics like the LA Times' "Pinpointing Pollution." It layers three things on a real street basemap: a pale dot texture showing modeled wildfire hazard exposure as background context (same hex-grid technique as the bivariate map earlier), a bubble for every major historical fire (≥3,000 acres) sized by acres burned, and color grouped by the decade it occurred. This makes two things visible at once that no earlier chart showed together: which parts of the county the *real* historical fires actually hit, and whether large fires have been getting more frequent in recent decades.

In [ ]:
import contextily as cx
from matplotlib.patches import Rectangle

map_df_3857 = map_df.to_crs(epsg=3857)

# Reuse the same fire perimeters already loaded above, just re-filter for "major" fires (>=3,000 acres)
bubble_fires = gpd.read_file(FIRE_CACHE, bbox=bbox).to_crs(epsg=3857)
bubble_fires = bubble_fires[bubble_fires['GIS_ACRES'] >= 3000].copy()
bubble_fires = bubble_fires.dropna(subset=['DECADES'])
bubble_fires['centroid'] = bubble_fires.geometry.centroid
print(f"Major fires (>=3,000 acres): {len(bubble_fires)}")

# --- Background texture: same hex-grid technique as the bivariate map, muted single hue ---
minx0, miny0, maxx0, maxy0 = map_df_3857.total_bounds
dxb = 1900
dyb = dxb * np.sqrt(3) / 2
xsb = np.arange(minx0 - dxb, maxx0 + dxb, dxb)
centers_b = []
row_b = 0
yb = miny0 - dyb
while yb < maxy0 + dyb:
    x_offset = (dxb / 2) if (row_b % 2) else 0
    for xb in xsb + x_offset:
        centers_b.append((xb, yb))
    yb += dyb
    row_b += 1
pts_b = gpd.GeoDataFrame(geometry=[Point(c) for c in centers_b], crs='EPSG:3857')
texture_pts = gpd.sjoin(pts_b, map_df_3857[['zip', 'wildfire_pct', 'geometry']], how='inner', predicate='within')
print(f"Texture points: {len(texture_pts)}")

# --- Tighter bounds based on where the content actually is, with extra room at top for the masthead ---
allx = np.concatenate([texture_pts.geometry.x.values, bubble_fires['centroid'].x.values])
ally = np.concatenate([texture_pts.geometry.y.values, bubble_fires['centroid'].y.values])
bminx, bmaxx = allx.min(), allx.max()
bminy, bmaxy = ally.min(), ally.max()
padx = (bmaxx - bminx) * 0.04
bminx, bmaxx = bminx - padx, bmaxx + padx
bminy = bminy - (bmaxy - bminy) * 0.03
bmaxy = bmaxy + (bmaxy - bminy) * 0.18

fig, ax = plt.subplots(figsize=(9, 12.5), dpi=200)
fig.patch.set_facecolor('white')

texture_norm = Normalize(vmin=0, vmax=texture_pts['wildfire_pct'].quantile(0.95))
tex_sizes = 2 + texture_norm(texture_pts['wildfire_pct'].clip(upper=texture_pts['wildfire_pct'].quantile(0.95))) * 10
ax.scatter(texture_pts.geometry.x, texture_pts.geometry.y, s=tex_sizes, color='#8fa8c9', alpha=0.35, linewidths=0, zorder=2)

ax.set_xlim(bminx, bmaxx)
ax.set_ylim(bminy, bmaxy)
ax.set_aspect('equal')
ax.axis('off')

cx.add_basemap(ax, source=cx.providers.CartoDB.Positron, zorder=1)

# White masthead panel so the headline/subtitle never overlaps basemap labels underneath
masthead_h = (bmaxy - bminy) * 0.16
ax.add_patch(Rectangle((bminx, bmaxy - masthead_h), bmaxx - bminx, masthead_h,
                        facecolor='white', edgecolor='none', zorder=3))

# --- Foreground bubbles: one per major fire, size = acres burned, color = decade ---
decade_order = sorted(bubble_fires['DECADES'].unique())
decade_cmap = LinearSegmentedColormap.from_list(
    'decade', ['#f7ecd1', '#f7c948', '#f2822a', '#e8412c', '#c31e5c', '#5b1a8c']
)
decade_norm = Normalize(vmin=0, vmax=len(decade_order) - 1)
decade_to_color = {d: decade_cmap(decade_norm(i)) for i, d in enumerate(decade_order)}

bubble_sizes = np.sqrt(bubble_fires['GIS_ACRES']) * 1.1
ax.scatter(
    bubble_fires['centroid'].x, bubble_fires['centroid'].y, s=bubble_sizes,
    color=[decade_to_color[d] for d in bubble_fires['DECADES']],
    alpha=0.8, edgecolors='white', linewidths=0.6, zorder=4,
)

# --- Masthead text ---
title_y = bmaxy - (bmaxy - bminy) * 0.015
ax.text(bminx + padx * 0.3, title_y, 'FOCUS  |  WILDFIRE RISK', fontsize=10.5, color='#b3401f',
        fontweight='bold', va='top', family='sans-serif', zorder=5)
ax.text(bminx + padx * 0.3, title_y - (bmaxy - bminy) * 0.028, 'PINPOINTING FIRE EXPOSURE',
        fontsize=23, color='#1a1a1a', fontweight='bold', va='top', family='serif', zorder=5)
subtitle = ("Every circle is a major wildfire (≥ 3,000 acres) that has burned in or near\n"
            "LA County since 1950. Size shows acres burned; color shows the decade it occurred.\n"
            "The pale dot texture shows modeled wildfire hazard exposure by ZIP code.")
ax.text(bminx + padx * 0.3, title_y - (bmaxy - bminy) * 0.075, subtitle, fontsize=9.3,
        color='#4a4a4a', va='top', family='sans-serif', linespacing=1.5, zorder=5)

# --- Legends ---
leg_y = bminy + (bmaxy - bminy) * 0.045
leg_x0 = bminx + padx * 0.3
for i, d in enumerate(decade_order):
    lx = leg_x0 + i * (bmaxx - bminx) * 0.083
    ax.scatter([lx], [leg_y], s=90, color=decade_to_color[d], edgecolors='white', linewidths=0.6, zorder=5)
    ax.text(lx, leg_y - (bmaxy - bminy) * 0.02, d, fontsize=6.3, color='#4a4a4a', ha='center', va='top')

size_leg_x = bminx + (bmaxx - bminx) * 0.80
for acres, dy_off in [(5000, 0), (20000, 0.028), (80000, 0.062)]:
    r = np.sqrt(acres) * 1.1
    ax.scatter([size_leg_x], [leg_y + (bmaxy - bminy) * dy_off - (bmaxy - bminy) * 0.02],
               s=r, facecolor='none', edgecolor='#4a4a4a', linewidths=0.8, zorder=5)
ax.text(size_leg_x + (bmaxx - bminx) * 0.03, leg_y + (bmaxy - bminy) * 0.075,
        'Circle size =\nacres burned', fontsize=6.8, color='#4a4a4a', va='top')

plt.savefig("la_wildfire_bubble_map.png", bbox_inches='tight', facecolor='white')
plt.show()

What's informative about this view: Putting real fires on a real street basemap makes the risk tangible in a way a choropleth can't — you can see individual large fires sitting right up against named cities and neighborhoods (Santa Clarita, Simi Valley, Thousand Oaks) rather than abstract ZIP polygons. The decade coloring also surfaces a pattern the rest of the notebook doesn't show explicitly: large (≥3,000 acre) fires cluster heavily in the most recent two decades (2010s–2020s, the purple/magenta bubbles) along the same western and northern edges where our wildfire exposure and home-value layers already showed the highest overlap.

What could be improved: The 3,000-acre cutoff is somewhat arbitrary — chosen to keep the bubble count legible rather than derived from any threshold in the data. Bubble maps also suffer from overplotting in dense fire zones (e.g., the Santa Clarita/Angeles National Forest cluster), where overlapping circles make it hard to judge individual fire size. A version with slight jitter or a "top 20 fires" filter (labeled by name, like Woolsey and Palisades) would likely communicate the story faster in a live 15-minute presentation than the full 234-fire view.

Interactive Version: The Same Bubble Map, But Live

The static version above is presentation-ready as a single image, but it can't be explored — you can't zoom into the Santa Clarita fire cluster, hover a bubble to check which fire it actually is, or turn off the texture layer to declutter. This is the same masthead/texture/bubble concept rebuilt in Folium: real map tiles instead of a static basemap image, independently toggleable texture and fire layers, and hover tooltips (fire name, year, acres) on every bubble.

In [ ]:
interactive_fires = gpd.read_file(FIRE_CACHE, bbox=bbox).to_crs(epsg=4326)
interactive_fires = interactive_fires[interactive_fires['GIS_ACRES'] >= 3000].copy()
interactive_fires = interactive_fires.dropna(subset=['DECADES'])
interactive_fires['acres_fmt'] = interactive_fires['GIS_ACRES'].round(0).astype(int)
interactive_fires['YEAR_'] = interactive_fires['YEAR_'].astype('Int64')
interactive_fires['lat'] = interactive_fires.geometry.centroid.y
interactive_fires['lon'] = interactive_fires.geometry.centroid.x
print(f"Major fires (interactive): {len(interactive_fires)}")

# --- Background texture (hex-grid points, muted) ---
map_df_3310 = map_df.to_crs(epsg=3310)
minx, miny, maxx, maxy = map_df_3310.total_bounds
dxi = 2400
dyi = dxi * np.sqrt(3) / 2
xsi = np.arange(minx - dxi, maxx + dxi, dxi)
centers_i = []
row_i2 = 0
yi = miny - dyi
while yi < maxy + dyi:
    x_offset = (dxi / 2) if (row_i2 % 2) else 0
    for xi in xsi + x_offset:
        centers_i.append((xi, yi))
    yi += dyi
    row_i2 += 1
pts_i = gpd.GeoDataFrame(geometry=[Point(c) for c in centers_i], crs='EPSG:3310')
joined_i = gpd.sjoin(pts_i, map_df_3310[['zip', 'wildfire_pct', 'geometry']], how='inner', predicate='within')
tex_pts = joined_i.set_geometry('geometry').set_crs('EPSG:3310').to_crs(epsg=4326)
tex_pts['lat'] = tex_pts.geometry.y
tex_pts['lon'] = tex_pts.geometry.x
print(f"Texture points (interactive): {len(tex_pts)}")

bubble_map = folium.Map(location=[34.05, -118.25], zoom_start=9, tiles=None, prefer_canvas=True)
folium.TileLayer(
    tiles='cartodbpositron', name='Streets Basemap (CARTO)',
    attr='Basemap by CARTO, data by OpenStreetMap contributors', control=True,
).add_to(bubble_map)

texture_layer = folium.FeatureGroup(name='Wildfire Exposure (background texture)', show=True, overlay=True)
tex_norm_i = Normalize(vmin=0, vmax=tex_pts['wildfire_pct'].quantile(0.95))
for _, r in tex_pts.iterrows():
    v = min(r['wildfire_pct'], tex_pts['wildfire_pct'].quantile(0.95))
    radius = 1.2 + tex_norm_i(v) * 3.5
    folium.CircleMarker(
        location=(r['lat'], r['lon']), radius=radius, color=None, fill=True,
        fill_color='#5b7ca8', fill_opacity=0.35, weight=0,
    ).add_to(texture_layer)
texture_layer.add_to(bubble_map)

fires_layer = folium.FeatureGroup(name='Major Fires, 1950-2025 (>=3,000 acres)', show=True, overlay=True)
for _, r in interactive_fires.iterrows():
    radius = max(3, np.sqrt(r['GIS_ACRES']) * 0.055)
    popup_html = f"<b>{r['FIRE_NAME']}</b> ({int(r['YEAR_'])})<br>{r['acres_fmt']:,} acres burned"
    folium.CircleMarker(
        location=(r['lat'], r['lon']), radius=radius,
        color='#b3401f', weight=1,
        fill=True, fill_color='#f2d9a1', fill_opacity=0.85,
        popup=folium.Popup(popup_html, max_width=220),
        tooltip=f"{r['FIRE_NAME']} ({int(r['YEAR_'])}) — {r['acres_fmt']:,} acres",
    ).add_to(fires_layer)
fires_layer.add_to(bubble_map)

# --- Masthead banner (fixed HTML overlay, mirrors the static version's title block) ---
masthead_html = '''
<div style="position: fixed; top: 0; left: 0; right: 0; z-index: 9999;
            background: white; padding: 14px 20px 10px 20px;
            box-shadow: 0 2px 6px rgba(0,0,0,0.15); font-family: sans-serif;">
  <div style="color:#b3401f; font-weight:bold; font-size:12px; letter-spacing:0.5px;">FOCUS &nbsp;|&nbsp; WILDFIRE RISK</div>
  <div style="font-family: Georgia, serif; font-weight:bold; font-size:24px; color:#1a1a1a; margin-top:2px;">
    Pinpointing Fire Exposure
  </div>
  <div style="font-size:12.5px; color:#4a4a4a; margin-top:4px; max-width:760px; line-height:1.45;">
    Every circle is a major wildfire (&ge; 3,000 acres) that has burned in or near LA County since 1950.
    Size shows acres burned. Toggle the wildfire-exposure texture and fire layers independently,
    and hover any fire for its name, year, and acreage.
  </div>
</div>
'''
bubble_map.get_root().html.add_child(folium.Element(masthead_html))
bubble_map.get_root().html.add_child(folium.Element(
    "<style>.folium-map { margin-top: 128px; height: calc(100% - 128px) !important; }</style>"
))

# --- Legend (fixed corner) ---
legend_html = '''
<div style="position: fixed; bottom: 30px; left: 30px; z-index: 9999;
            background: white; padding: 10px 14px; border-radius: 6px;
            box-shadow: 0 1px 6px rgba(0,0,0,0.3); font-size: 11.5px; line-height: 1.6;
            font-family: sans-serif;">
<span style="display:inline-block;width:10px;height:10px;border-radius:50%;
             background:#f2d9a1;border:1px solid #b3401f;margin-right:4px;"></span>
Historical major fire (size = acres burned)<br>
<span style="display:inline-block;width:10px;height:10px;border-radius:50%;
             background:#5b7ca8;opacity:0.5;margin-right:4px;"></span>
Wildfire exposure (background texture)
</div>
'''
bubble_map.get_root().html.add_child(folium.Element(legend_html))

folium.LayerControl(collapsed=False).add_to(bubble_map)
bubble_map.save("la_wildfire_bubble_map.html")
print("Saved to la_wildfire_bubble_map.html — open it directly in your browser")
bubble_map

What's informative about this view: This is the closest thing in the whole notebook to a finished "product" rather than an analysis chart — a masthead, a legend, and two independently toggleable layers a user can actually manipulate live during the presentation. Turning off the texture layer and zooming into the Woolsey/Palisades area makes the individual fire bubbles and their popups easy to read without the background competing for attention, which the static PNG can't do.

What could be improved: Rendering ~1,974 texture points and 234 fire bubbles as individual Leaflet CircleMarkers makes this noticeably heavier than the earlier interactive maps (both in file size and browser responsiveness when panning). If this becomes the main demo piece, switching the texture layer to a single rasterized image overlay (rendered once with matplotlib, then placed with `folium.raster_layers.ImageOverlay`) would cut the marker count dramatically while keeping the same look.

A More Professional Version: The Same Data, Dressed for a Real Estate Audience

The dot-textured Investment Explorer above is visually distinctive, but it reads more like a data-journalism infographic than something a real estate firm would trust for underwriting-adjacent decisions. This version keeps the exact same underlying data and layers (wildfire risk, income, home value, appreciation, Investment Score, Investment Category, top picks) but restyles it for that audience:

- **Smooth ZIP-boundary choropleth fills** instead of a hex-dot texture — the boundary-shading style used by Zillow, Redfin, and Realtor.com, rather than a grid of individual dots.
- **A restrained, "trustworthy" palette** — muted blues for value/growth metrics, a desaturated red for risk, and navy/teal/burgundy/gray (instead of cream/orange/purple/red) for the Investment Category, closer to what a financial dashboard would use than an editorial chart.
- **Cleaner typography and markers** — a sans-serif masthead instead of a serif headline, and simple navy-and-gold numbered circles for the Top Picks instead of teardrop map pins.
- The same JS-based layer-exclusivity fix from the Investment Explorer, applied from the start this time, so the street basemap never disappears when switching layers.

- **Usability-driven polish, based on our own usability study**: a single legend panel that now updates automatically to match whichever layer is active (previously only the first layer had a visible color scale); an ⓘ info toggle next to "Investment Opportunity Score" that expands a plain-language explanation of the formula in place, since testers could see the number but not explain how it was derived; a more prominent, labeled layer control ("MAP LAYERS — TOGGLE TO EXPLORE") since some testers missed it at its original size; a ZIP/neighborhood search box (top right); and click-to-open popups on the ZIP polygons themselves, not just the numbered pins, since testers expected the polygons to be directly interactive.

- **Usability-driven polish, based on our own usability study**: a single legend panel that now updates automatically to match whichever layer is active (previously only the first layer had a visible color scale); an ⓘ info toggle next to "Investment Opportunity Score" that expands a plain-language explanation of the formula in place, since testers could see the number but not explain how it was derived; a more prominent, labeled layer control ("MAP LAYERS — TOGGLE TO EXPLORE") since some testers missed it at its original size; a ZIP/neighborhood search box (top right); and click-to-open popups on the ZIP polygons themselves, not just the numbered pins, since testers expected the polygons to be directly interactive.

In [ ]:
import json as _json
import numpy as np
import matplotlib as mpl
from folium.features import GeoJsonPopup
from folium.plugins import Search

# Soft/pastel palette, chosen to read as light and approachable rather than "financial
# terminal," while still reusing the same color logic as the continuous layers: blue tones
# for the good outcomes (matches the Blues palette used for value/growth), coral/red for the
# bad outcome (matches the Reds palette used for wildfire risk), and a lavender blend of the
# two for the "mixed" quadrant (real risk, but real growth too) — so the mapping stays
# intuitive even for someone who never opens the legend.
category_colors_pro = {
    'Opportunity Zone (high risk, high appreciation)': '#B9A8DC',
    'Red Flag (high risk, low appreciation)': '#F0A8A0',
    'Safe Bet (low risk, high appreciation)': '#A8CDEB',
    'Low Priority (low risk, low appreciation)': '#DAD6C8',
}

# Plain-language labels shown to users (legend, popups) — the dict keys above stay as the
# original technical strings since those are what's actually stored in map_df/GeoJson
# properties and used for style lookups; only the *display* text changes.
category_display = {
    'Opportunity Zone (high risk, high appreciation)': 'Opportunity Zone — risky, but growing fast',
    'Red Flag (high risk, low appreciation)': 'Red Flag — risky, without the growth',
    'Safe Bet (low risk, high appreciation)': 'Safe Bet — low risk, growing fast',
    'Low Priority (low risk, low appreciation)': 'Low Priority — low risk, slow growth',
}

pro_map = folium.Map(location=[34.05, -118.25], zoom_start=9, tiles=None, prefer_canvas=True)
folium.TileLayer(
    tiles='cartodbpositron', name='Streets Basemap (CARTO)',
    attr='Basemap by CARTO, data by OpenStreetMap contributors', control=True,
).add_to(pro_map)

# --- Typography: Cormorant Garamond, a delicate high-contrast serif in the same family used
# by fashion/editorial luxury brands, for headlines and numerals — set at 600/700 weight since
# its thin default weight gets lost at small sizes — alongside a clean Inter for body copy and
# Leaflet's own UI chrome, in place of the plain system 'Helvetica Neue' stack used before. ---
pro_map.get_root().html.add_child(folium.Element(
    "<link rel=\"preconnect\" href=\"https://fonts.googleapis.com\">"
    "<link rel=\"preconnect\" href=\"https://fonts.gstatic.com\" crossorigin>"
    "<link href=\"https://fonts.googleapis.com/css2?family=Cormorant+Garamond:wght@500;600;700&family=Inter:wght@400;500;600&display=swap\" rel=\"stylesheet\">"
))
pro_map.get_root().html.add_child(folium.Element(
    "<style>.leaflet-container { font-family: 'Inter', 'Helvetica Neue', Arial, sans-serif; }</style>"
))

# --- Metric definitions: (column, label, color palette, value formatter for the legend) ---
def _fmt_currency(v): return f"${v:,.0f}"
def _fmt_percent0(v): return f"{v:.0f}%"
def _fmt_appreciation(v): return f"{v * 100:.0f}%"
def _fmt_score(v): return f"{v:.0f}"

metrics_pro = [
    ('wildfire_pct', 'Wildfire Risk', 'Reds', _fmt_percent0),
    ('median_household_income', 'Household Income', 'Blues', _fmt_currency),
    ('current_home_value', 'Home Value', 'Blues', _fmt_currency),
    ('appreciation_10yr', 'Price Growth (Last 10 Years)', 'Blues', _fmt_appreciation),
    ('investment_score', 'Investment Score', 'PuBu', _fmt_score),
]

exclusive_names = []
for i, (col, label, palette, fmt) in enumerate(metrics_pro):
    choropleth = folium.Choropleth(
        geo_data=map_df,
        data=map_df,
        columns=['zip', col],
        key_on='feature.properties.zip',
        fill_color=palette,
        fill_opacity=0.82,
        line_color='#5b6672',
        line_weight=0.6,
        line_opacity=0.5,
        legend_name=label,
        name=label,
        show=(i == 0),
        overlay=True,
        nan_fill_color='#e5e2da',
    ).add_to(pro_map)
    exclusive_names.append(choropleth.get_name())
    # Folium's built-in colorbar legend isn't tied to the layer's own show/hide toggle — it's
    # always rendered regardless of which overlay is checked, so with 5 metrics all 5 default
    # legends would stack on top of each other. Strip all of them and drive a single custom
    # legend panel from JS instead (see legend_configs / #dashLegend below), which *does* update
    # with whichever layer is actually active.
    for key in list(choropleth._children):
        if key.startswith('color_map'):
            del choropleth._children[key]

# Investment Type as a smooth polygon fill, not a dot texture. Pastel fills read lighter than
# the continuous-metric choropleths, so they get a touch more opacity (0.88 vs 0.82) to stay
# visible against the basemap rather than washing out.
cat_layer = folium.GeoJson(
    map_df,
    style_function=lambda f: {
        'fillColor': category_colors_pro[f['properties']['investment_category']],
        'fillOpacity': 0.88,
        'color': '#8b8f99',
        'weight': 0.6,
    },
    name='Investment Type',
    show=False,
    overlay=True,
).add_to(pro_map)
exclusive_names.append(cat_layer.get_name())

# --- Custom legend content, one entry per exclusive layer above (same order) ---
def _gradient_css(palette, n=6):
    cmap = mpl.colormaps[palette]
    stops = [mpl.colors.to_hex(cmap(t)) for t in np.linspace(0.08, 0.92, n)]
    return 'linear-gradient(to right, ' + ', '.join(stops) + ')'

legend_configs = []
for col, label, palette, fmt in metrics_pro:
    legend_configs.append({
        'type': 'gradient',
        'title': label,
        'gradient': _gradient_css(palette),
        'min': fmt(map_df[col].min()),
        'max': fmt(map_df[col].max()),
    })
legend_configs.append({
    'type': 'category',
    'title': 'Investment Type',
    'swatches': [{'color': c, 'label': category_display[k]} for k, c in category_colors_pro.items()],
})

# --- Search field: lets the search box below match on either ZIP or city/neighborhood name ---
map_df['search_key'] = map_df['zip'].astype(str) + ' — ' + map_df['City'].astype(str)
# Plain-language, pre-formatted versions of the raw numbers for the tooltip/popup — a whole
# number ("14") reads easier than a decimal score, and a "%"-suffixed string reads easier
# than a bare fraction like "0.234" (which doesn't mean anything at a glance without already
# knowing the column is a fraction).
map_df['investment_score_fmt'] = map_df['investment_score'].round(0).astype(int)
map_df['appreciation_10yr_fmt'] = (map_df['appreciation_10yr'] * 100).round(1).astype(str) + '%'
map_df['investment_category_display'] = map_df['investment_category'].map(category_display)

# Hover tooltip + click popup: property-card style, same info both ways so the map works
# the same whether a user hovers (desktop) or taps (mobile/touch).
zip_info_layer = folium.GeoJson(
    map_df,
    style_function=lambda x: {'fillOpacity': 0, 'color': 'transparent'},
    tooltip=GeoJsonTooltip(
        fields=['zip', 'City', 'investment_category_display', 'current_home_value',
                'appreciation_10yr_fmt', 'wildfire_pct', 'investment_score_fmt'],
        aliases=['ZIP:', 'Neighborhood:', 'Investment Type:', 'Home Value ($):',
                 'Price Growth (10 yrs):', 'Wildfire Risk (%):', 'Investment Score:'],
        localize=True,
    ),
    popup=GeoJsonPopup(
        fields=['zip', 'City', 'investment_category_display', 'current_home_value',
                'appreciation_10yr_fmt', 'wildfire_pct', 'investment_score_fmt'],
        aliases=['ZIP:', 'Neighborhood:', 'Investment Type:', 'Home Value ($):',
                 'Price Growth (10 yrs):', 'Wildfire Risk (%):', 'Investment Score:'],
        localize=True,
    ),
    name='ZIP details (hover / click)',
    overlay=True,
    control=True,
).add_to(pro_map)

# ZIP / neighborhood search box, top-right — matches on ZIP digits or city name
Search(
    layer=zip_info_layer,
    search_label='search_key',
    placeholder='Search ZIP or neighborhood…',
    collapsed=False,
    position='topright',
).add_to(pro_map)

# City labels
city_labels = {
    'Malibu': (-118.7798, 34.0259), 'Santa Monica': (-118.4912, 34.0195),
    'Downtown LA': (-118.2437, 34.0407), 'Pasadena': (-118.1445, 34.1478),
    'Long Beach': (-118.1937, 33.7701), 'Santa Clarita': (-118.5426, 34.3917),
}
for name, (lon, lat) in city_labels.items():
    folium.Marker(
        location=(lat, lon),
        icon=folium.DivIcon(html=f'<div style="font-size:11px;color:#2b2b2b;font-weight:600;white-space:nowrap;">{name}</div>'),
    ).add_to(pro_map)

# Top Investment Picks: flat pastel-green numbered badges — same lightness/saturation family
# as the 4 Investment Type swatches (lavender/coral/sky-blue/sand) rather than the earlier
# saturated gold or jewel-toned emerald, so the pins read as part of the same soft palette
# instead of a separate "special" material. A true, less blue-leaning pastel green doesn't collide with any hue used
# elsewhere (Reds/Blues/PuBu metric palettes, or the lavender/coral/sky-blue/sand category
# colors), and the ivory-then-navy double ring — the same contrast trick as before — keeps it
# legible against every fill color underneath even at this lighter, flatter weight.
#
# The numeral itself uses Inter, not Cormorant Garamond: tested up close, Cormorant's "1" is
# just a bare vertical stroke with no serif, easy to misread as a capital "I" in a small badge
# with no other context (the "#1" in the popup below is fine since the "#" disambiguates it,
# but a lone digit in a circle needs a face with unambiguous digit shapes).
top_picks = map_df.sort_values('investment_score', ascending=False).head(8).reset_index(drop=True)
top_picks['rank'] = top_picks.index + 1
for _, row in top_picks.iterrows():
    centroid = row.geometry.centroid
    pin_html = f"""
    <div style="
        width:29px; height:29px;
        background:#A6D896;
        border:2px solid #faf6ea;
        border-radius:50%;
        display:flex; align-items:center; justify-content:center;
        color:#1c3f5c; font-weight:700; font-size:13.5px;
        box-shadow:0 0 0 1.5px #1c3f5c, 0 2px 6px rgba(0,0,0,0.4);
        font-family: 'Inter', 'Helvetica Neue', Arial, sans-serif;">
        {row['rank']}
    </div>
    """
    popup_html = f"""
    <div style="font-family:'Inter', 'Helvetica Neue', Arial, sans-serif;">
    <div style="font-family:'Cormorant Garamond', Georgia, serif; font-weight:700; font-size:16px; color:#1c2a33;">
        <span style="color:#4c8c52;">#{row['rank']}</span> — {row['City']} ({row['zip']})
    </div>
    <span style="color:{category_colors_pro[row['investment_category']]};">&#9632;</span> {category_display[row['investment_category']]}<br>
    Home Value: ${row['current_home_value']:,.0f}<br>
    Price Growth (10 yrs): {row['appreciation_10yr'] * 100:.1f}%<br>
    Wildfire Risk: {row['wildfire_pct']:.1f}%<br>
    Investment Score: {row['investment_score']:.0f}
    </div>
    """
    folium.Marker(
        location=[centroid.y, centroid.x],
        icon=folium.DivIcon(html=pin_html, icon_size=(29, 29), icon_anchor=(14.5, 14.5)),
        popup=folium.Popup(popup_html, max_width=250),
        tooltip=f"#{row['rank']}: {row['City']} — Investment Score {row['investment_score']:.0f}",
    ).add_to(pro_map)

# --- Masthead: clean sans-serif, corporate palette, with a plain-language score explainer ---
# Usability testing found users could see the Investment Score number but not explain how it
# was derived (multiple participants asked "how exactly is this calculated?"), and more broadly
# that the whole dashboard read as too technical for someone who isn't already fluent in real
# estate/finance terms ("10-year price appreciation," "percentile," etc.). This rewrite drops
# jargon wherever the underlying idea can be said in plain English, and the ⓘ toggle explains
# the score in one click, right where it's first mentioned, instead of a dense paragraph nobody
# reads to the end.
#
# The score itself is a simplified version of "risk-adjusted return," the same idea real estate
# investors use to compare a property's likely payoff against how risky it is (institutional
# versions typically divide return by volatility/a Sharpe-ratio-style denominator, using data
# like vacancy rates and loan coverage ratios we don't have here). We approximate it by ranking
# each ZIP's price growth and wildfire risk against every other ZIP, then taking growth-rank
# minus risk-rank — same spirit (reward relative to risk), simplified to data we can actually
# verify, and described honestly below as a screening tool rather than a professional appraisal.
masthead_html = '''
<div id="dashMasthead" style="position: fixed; top: 0; left: 0; right: 0; z-index: 9999;
            background: white; padding: 14px 20px 10px 20px;
            box-shadow: 0 2px 6px rgba(0,0,0,0.12); font-family: 'Inter', 'Helvetica Neue', Arial, sans-serif;
            border-bottom: 3px solid #1c3f5c;">
  <div style="color:#4c8c52; font-weight:600; font-size:11.5px; letter-spacing:1.4px;">LA COUNTY MARKET INTELLIGENCE</div>
  <div style="font-family:'Cormorant Garamond', Georgia, 'Times New Roman', serif; font-weight:700; font-size:30px; color:#1c2a33; margin-top:2px; letter-spacing:0.3px;">
    Wildfire Risk &amp; Investment Dashboard
  </div>
  <div style="font-family:'Cormorant Garamond', Georgia, serif; font-weight:600; font-style:italic; font-size:15px; color:#3a4a56; margin-top:2px;">
    Which LA County ZIP codes are still worth the wildfire risk?
  </div>
  <div id="dashMastheadDesc" style="font-size:12px; color:#5b6672; margin-top:5px; max-width:780px; line-height:1.5;">
    Every ZIP code gets an <b>Investment Score</b>
    <span id="scoreInfoBtn" onclick="var p=document.getElementById('scoreInfoPanel'); p.style.display=(p.style.display==='none'?'block':'none');"
          style="cursor:pointer; display:inline-flex; align-items:center; justify-content:center;
                 width:15px; height:15px; border-radius:50%; background:#1c3f5c; color:white;
                 font-size:10.5px; font-weight:700; vertical-align:middle; margin-left:2px;"
          title="What is this?">i</span>
    that balances price growth against fire risk, so you can spot areas doing well despite the
    danger. The 8 pastel-green markers are our top picks. Search a ZIP or neighborhood above, or explore
    the layers below.
    <div id="scoreInfoPanel" style="display:none; margin-top:6px; padding:8px 10px; background:#f3f1ec;
                border-left:3px solid #1c3f5c; border-radius:3px; max-width:640px;">
      <b style="font-family:'Cormorant Garamond', Georgia, serif; font-weight:700; font-size:15px;">What is the Investment Score?</b> Real estate investors often weigh a property's likely
      growth against how risky it is — this score uses that same idea, simplified. Every ZIP code
      is compared to every other ZIP code in two ways: how much home prices grew over the last 10
      years, and how much wildfire danger it has. A ZIP that grew a lot <i>and</i> has low fire
      risk scores highest; one that barely grew <i>and</i> has high fire risk scores lowest.
      It's a quick way to compare neighborhoods side by side — not a prediction, an appraisal, or
      financial advice. Always do your own research (and talk to a real estate professional)
      before making a decision.
    </div>
  </div>
</div>
'''
pro_map.get_root().html.add_child(folium.Element(masthead_html))
# Fallback margin (matches the masthead's typical desktop height) so the map isn't hidden
# under the masthead for the instant before the JS below measures the real, rendered height —
# JS then overrides this to fit whatever height the masthead actually wraps to (mobile-width
# screens wrap the description onto several more lines than desktop, so a fixed pixel value
# either leaves a gap on desktop or overlaps the layer control on narrow phones).
pro_map.get_root().html.add_child(folium.Element('''
<style>
.folium-map { margin-top: 128px; height: calc(100% - 128px) !important; }
@media (max-width: 700px) {
  #dashMasthead { padding: 10px 14px 8px 14px; }
  #dashMastheadDesc { font-size: 11px !important; line-height: 1.3 !important; }
}
</style>
'''))

# --- Legend: single panel, content swapped by JS to match whichever layer is active ---
legend_html = '''
<div id="dashLegend" style="position: fixed; bottom: 30px; left: 30px; z-index: 9999;
            background: white; padding: 12px 15px; border-radius: 4px;
            box-shadow: 0 1px 6px rgba(0,0,0,0.25); font-size: 12px; line-height: 1.6;
            font-family: 'Inter', 'Helvetica Neue', Arial, sans-serif; border-left: 3px solid #1c3f5c;
            min-width: 200px; max-width: 240px;">
</div>
'''
pro_map.get_root().html.add_child(folium.Element(legend_html))

# --- Layer control: same widget, made harder to miss (usability testing found some
# participants overlooked it entirely at default size/contrast) ---
layer_control_css = '''
<style>
.leaflet-control-layers {
  border: 2px solid #1c3f5c !important;
  box-shadow: 0 3px 12px rgba(0,0,0,0.3) !important;
  padding: 8px 12px !important;
}
.leaflet-control-layers-list { font-size: 13px !important; }
.leaflet-control-layers-overlays label { padding: 2px 0; }
</style>
'''
pro_map.get_root().html.add_child(folium.Element(layer_control_css))
folium.LayerControl(collapsed=False).add_to(pro_map)

# --- Wiring: layer exclusivity (only one metric/category visible at a time), legend swap,
# and a "MAP LAYERS" header prepended to the layer control so it reads as an instruction,
# not just an unlabeled checkbox list. ---
main_js = f"""
<script>
var legendConfigs = {_json.dumps(legend_configs)};

var legendTitleStyle = "font-family:'Cormorant Garamond',Georgia,serif; font-weight:700; font-size:17px; color:#1c2a33;";

function renderLegend(cfg) {{
    var el = document.getElementById('dashLegend');
    if (!el) return;
    var title = '<div style="' + legendTitleStyle + '">' + cfg.title + '</div>';
    if (cfg.type === 'gradient') {{
        el.innerHTML = title +
            '<div style="margin:6px 0 4px 0; height:10px; border-radius:3px; background:' + cfg.gradient + ';"></div>' +
            '<div style="display:flex; justify-content:space-between; color:#5b6672;">' +
            '<span>' + cfg.min + '</span><span>' + cfg.max + '</span></div>' +
            '<i style="color:#8a8378; display:block; margin-top:6px;">Color scale updates with the active layer</i>';
    }} else {{
        var rows = cfg.swatches.map(function(s) {{
            return '<span style="color:' + s.color + ';">&#9632;</span> ' + s.label + '<br>';
        }}).join('');
        el.innerHTML = title + rows +
            '<i style="color:#8a8378;">(only shown when that layer is active)</i>';
    }}
}}

window.addEventListener('load', function() {{
(function() {{
    var exclusiveLayers = [{', '.join(exclusive_names)}];
    var map = {pro_map.get_name()};

    // Drive layer exclusivity from the checkbox clicks directly, rather than Leaflet's
    // 'overlayadd'/'overlayremove' events. Those events fire once per individual polygon
    // sub-layer in a Choropleth's FeatureGroup as well as once for the group itself, and
    // reacting to them by calling addLayer/removeLayer from inside their own handler set
    // off a re-entrant cascade that could leave two layers on (or the wrong one) depending
    // on click timing. Listening for the checkbox's own click event, after the browser's
    // default toggle and Leaflet's own add/remove for *that one* checkbox have already run,
    // avoids the cascade entirely: we just clean up every other exclusive layer afterward.
    var overlayCheckboxes = document.querySelectorAll('.leaflet-control-layers-overlays input[type=checkbox]');

    overlayCheckboxes.forEach(function(cb, idx) {{
        if (idx >= exclusiveLayers.length) return;
        cb.addEventListener('click', function() {{
            if (!cb.checked) return;
            exclusiveLayers.forEach(function(layer, i) {{
                if (i === idx) return;
                if (map.hasLayer(layer)) map.removeLayer(layer);
                overlayCheckboxes[i].checked = false;
            }});
            renderLegend(legendConfigs[idx]);
        }});
    }});

    renderLegend(legendConfigs[0]);

    var overlaysBlock = document.querySelector('.leaflet-control-layers-overlays');
    if (overlaysBlock) {{
        var header = document.createElement('div');
        header.innerHTML = '<b style="font-size:10.5px; letter-spacing:1.2px; color:#4c8c52;">MAP LAYERS — TOGGLE TO EXPLORE</b><hr style="margin:5px 0 7px 0; border:none; border-top:1px solid #dde3dd;">';
        overlaysBlock.parentNode.insertBefore(header, overlaysBlock);
    }}

    // The 7th checkbox ("ZIP details") isn't part of the exclusivity group above — it's an
    // always-on overlay independent of whichever metric/category is picked. Without a visual
    // cue it looks identical to the other 6, which invites the reasonable but wrong guess that
    // unchecking a metric would also turn it off. A thin divider + one-line caption makes the
    // difference explicit instead of leaving it to be discovered by trial and error.
    var zipDetailsCheckbox = overlayCheckboxes[exclusiveLayers.length];
    if (zipDetailsCheckbox) {{
        var zipLabel = zipDetailsCheckbox.closest('label');
        if (zipLabel) {{
            var note = document.createElement('div');
            note.innerHTML = '<hr style="margin:6px 0 5px 0; border:none; border-top:1px solid #dde3dd;">' +
                '<div style="font-size:9.5px; color:#8a8f8a; margin-bottom:2px;">Always available, independent of the layer above:</div>';
            zipLabel.parentNode.insertBefore(note, zipLabel);
        }}
    }}

    // Fit the map to however tall the masthead actually rendered, instead of the fixed
    // 128px fallback in the <style> tag above — on a narrow/mobile screen the description
    // wraps onto several extra lines, so a hardcoded height either leaves a gap (desktop)
    // or lets the layer control overlap the masthead text (mobile).
    var mapDiv = document.querySelector('.folium-map');
    function fitMastheadHeight() {{
        var masthead = document.getElementById('dashMasthead');
        if (!masthead || !mapDiv) return;
        var h = masthead.offsetHeight;
        mapDiv.style.marginTop = h + 'px';
        mapDiv.style.height = 'calc(100% - ' + h + 'px)';
        map.invalidateSize();
    }}
    fitMastheadHeight();
    window.addEventListener('resize', fitMastheadHeight);
}})();
}});
</script>
"""
pro_map.get_root().html.add_child(folium.Element(main_js))

pro_map.save("la_wildfire_professional_map.html")
print("Saved to la_wildfire_professional_map.html — open it directly in your browser")
pro_map

What's informative about this view: Side by side with the Investment Explorer, this makes the design tradeoff concrete rather than abstract — same data, same layers, same Investment Score and Top Picks logic, but a completely different read. The smooth choropleth immediately looks more like something from an MLS or a Zillow-style market report, and the muted navy/teal/burgundy category colors don't compete with each other the way the brighter palette did. For a presentation to an actual real estate audience, this is very likely the version to lead with; the hex-dot version is better suited to a design-process discussion of how we got there.

What could be improved: Smooth choropleths hide the underlying spatial resolution — a whole ZIP code shares one fill color, which can visually overstate homogeneity within large or diverse ZIPs (exactly the "West Hills labeled just Los Angeles" problem from earlier, just less visible now that there's no dot texture to hint at internal variation). A hybrid — smooth fills as the default, with the hex-dot texture available as an optional "detail" overlay — would let a user zoom into a ZIP and see that finer structure on demand rather than losing it. The color choices here also reflect one team's sense of "looks professional"; the honest test would be showing both versions to an actual real estate audience and asking which they'd trust more.

## Design Process & Change Log

*Group 5 — Amin Sharifi, Kala Fejzo, Moreki Khobotlo*

This section documents how this dashboard changed between our midterm submission and this final version, and why each change was made. The same write-up (plus a QA section on the bugs we found while testing) lives at [`design_process/DESIGN_CHANGELOG.md`](design_process/DESIGN_CHANGELOG.md); our usability study results are summarized in [`usability_study_summary.md`](usability_study_summary.md).

### Midterm screenshots, for reference

| Iteration 1 — static wildfire map (no investment framing yet) | Iteration 2 — hex-dot Investment Explorer (midterm) |
|---|---|
| ![Iteration 1](design_process/before_iteration1_static_wildfire_map.png) | ![Iteration 2](design_process/before_investment_explorer_hexdot.png) |

We also prototyped a third, visually distinct direction in Observable (a recovery-timeline focused layout, "Burning Value") that was not carried forward — the risk/appreciation/investment-score framing of the Investment Explorer tested better with our target users and became the basis for everything downstream.

### Current visualization, for reference

| Desktop | Mobile |
|---|---|
| ![Final dashboard, desktop](design_process/after_dashboard_desktop.png) | ![Final dashboard, mobile](design_process/after_dashboard_mobile.png) |

### Change log

| # | Before | After | Why |
|---|---|---|---|
| 1 | Iteration 1 had no investment framing at all — just historical fire locations sized by acres burned, with no way to compare neighborhoods as opportunities | Added an **Investment Score** (price growth weighed against wildfire risk) and 8 ranked "Top Picks" markers | Stronger decision support — turns "here's where fires happened" into "here's where you should actually look first" |
| 2 | Hex-dot texture map, bright editorial palette (cream/purple/orange/red) — reads like a data-journalism graphic | Smooth ZIP-boundary choropleth, closer to a Zillow/Redfin-style market map | More context and more trustworthy for the audience actually evaluating it — several usability testers described it in exactly those terms ("something Zillow or Redfin could build") |
| 3 | Color legend was only ever correct for the default layer (Wildfire Risk); switching to any other metric left a stale or missing legend — confirmed as a real bug present since the midterm build, not something introduced later | A single legend panel that re-renders to match whichever layer is actually active | Easier to interpret — a color scale that doesn't match what's on screen is worse than no legend, since it actively misleads instead of just being unhelpful |
| 4 | Dense, technical copy throughout: "10-Year Price Appreciation (Fraction)," "Investment Opportunity Score," percentile-rank formula spelled out in the header | Plain-language labels ("Price Growth," "Investment Score") and a one-click ⓘ explainer using everyday language | Directly answers our usability study's #1 finding — testers could see the score but not explain how it was derived, and said the dashboard needed to work "without me explaining it" |
| 5 | Dark, saturated "financial terminal" palette (navy/teal/burgundy/gray) for the Investment Category layer | Soft pastel palette (lavender/coral/sky-blue/sand) plus a matching pastel-green for the numbered pins | More approachable and friendlier on first impression, without losing legibility |
| 6 | No way to search for a specific ZIP; only the 8 numbered pins were clickable, ZIP polygons responded to hover only | Added a ZIP/neighborhood search box, made ZIP polygons clickable (not just the pins), and enlarged/labeled the layer control | Better exploration — testers specifically asked for a search bar and expected to be able to click directly on ZIP polygons |
| 7 | Default system sans-serif (Helvetica Neue) throughout, same as any generic web page | A serif/sans pairing (Cormorant Garamond for headlines and legend titles, Inter for body copy and data) | Gives the dashboard a more polished, boutique real-estate feel instead of reading as a generic corporate report |
| 8 | Dashboard presented six togglable layers with no framing — left it to the user to figure out what question they were supposed to be answering | Masthead now opens with one explicit question ("Which LA County ZIP codes are still worth the wildfire risk?") that every layer exists to help answer | Gives the visualization a single clear narrative instead of open-ended, undirected exploration |

## Usability Study Summary

*Group 5 — Amin Sharifi, Kala Fejzo, Moreki Khobotlo*

We ran a usability study on this dashboard covering three themes: Wildfire Risk Assessment, Housing Market Impact and Recovery, and Investment Decision Support. The full write-up (per-participant observations, comments, and a MoSCoW-prioritized list of findings) is at [`usability_study_summary.md`](usability_study_summary.md); a condensed one-slide version is at [`design_process/Usability_Study_Summary.pptx`](design_process/Usability_Study_Summary.pptx).

### Participants

**9 participants** tested the dashboard. Each self-selected one persona to role-play for their session, from: prospective homebuyer, homeowner, real estate investor, real estate professional, or policymaker/community planner.

### Tasks

- Identify the highest wildfire-risk ZIP
- Compare wildfire risk: Pasadena vs. Santa Monica
- Identify the highest-appreciation ZIP
- Identify the fastest-recovering neighborhood
- Explain what the Investment Score means
- Recommend one ZIP for long-term investment

### Main findings

- **Misunderstood** how the Investment Score was calculated (Participants 1, 6)
- **Overlooked** the layer controls at first (Participant 1)
- **Expected** ZIP polygons themselves to be clickable, not just the numbered pins (Participant 9)
- **Wanted** a search bar (Participant 3), plus more financial context in the popups — insurance costs, rental yield, historical appreciation, market inventory (Participants 2, 4, 8)
- **Strength**: participants found the dashboard visually professional and easy to interpret — one described it as "something Zillow or Redfin could build" (Participant 3)

Four of these findings (the ⓘ score explainer, the more prominent layer control, ZIP-polygon click support, and the search bar) are already addressed in the current dashboard — see the [Design Process & Change Log](#Design-Process-&-Change-Log) above.